# Capítulo 7 — Rolling returns

**Objetivo:** ver cómo cambia el CAGR en ventanas móviles de 3 y 5 años.

### Fórmula (ventana de $w$ meses)
$$\mathrm{CAGR}_{t}(w) = \left(\frac{V_t}{V_{t-w+1}}\right)^{12/(w-1)} - 1$$


### Calcular y graficar

In [1]:
import sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))
import portfolio_utils as pu

daily = pd.read_csv(ROOT/"data"/"precios_ajustados_diarios.csv", index_col=0, parse_dates=True)
monthly = pu.to_month_end(daily).loc[: "2026-09-30"]
bt = pu.backtest_portfolio(monthly, pu.CURRENT_WEIGHTS, start="2016-01-31", end="2026-09-30", rebalance="A")
spy = pu.backtest_portfolio(monthly, {"SPY":1}, start="2016-01-31", end="2026-09-30", rebalance="N")

roll3 = pu.rolling_cagr(bt["value"], 36)
roll5 = pu.rolling_cagr(bt["value"], 60)
roll3s = pu.rolling_cagr(spy["value"], 36)

fig, ax = plt.subplots(figsize=(11,5))
ax.plot(roll3.index, roll3*100, label="Portafolio 3a")
ax.plot(roll3s.index, roll3s*100, label="SPY 3a", alpha=0.8)
ax.plot(roll5.index, roll5*100, label="Portafolio 5a", ls="--")
ax.set_ylabel("CAGR rodante (%)")
ax.set_title("Retornos rodantes anualizados")
ax.legend(); plt.tight_layout()
plt.savefig(ROOT/"figures"/"05_rolling_returns.png", dpi=120)
plt.show()
print("Mín CAGR 3a portafolio:", f"{roll3.min()*100:.2f}%")
print("Máx CAGR 3a portafolio:", f"{roll3.max()*100:.2f}%")


![figura](../figures/05_rolling_returns.svg)


Mín CAGR 3a portafolio: 6.61%
Máx CAGR 3a portafolio: 33.86%


**Interpretación:** las ventanas que incluyen 2022 muestran el suelo; las que capturan 2023–2024 el techo. Un inversor que entre en el pico puede vivir años de CAGR rodante mediocre aun si el full-period luce excelente.

## Conclusiones
- El CAGR de todo el periodo **no** representa la experiencia de todos los horizontes de entrada.
- Comparar rolling vs SPY revela cuándo el sesgo activo aportó de verdad.

## Ejercicios propuestos
1. Calcula rolling de 12 meses (más ruidoso) y comenta.
2. ¿En qué % de meses el CAGR 3a del portafolio superó al de SPY?
3. Relaciona el mínimo rolling 3a con la fecha del Max DD.
